# Chronological History and Feature Engineering

Loads the frozen Step 2 feature contract and one small real Parquet batch to demonstrate ordering, history fields, and leakage controls.

**Safety:** default execution is lightweight. Expensive full-data work runs only behind an explicit environment flag.

In [1]:
from pathlib import Path
import os
import json
import pandas as pd

WORKSPACE = Path(
    r"C:\Users\Monil A\Desktop\Research Documents\ASSISTments Dataset"
)

if not WORKSPACE.is_dir():
    raise FileNotFoundError(
        f"ASSISTments research workspace not found: {WORKSPACE}"
    )

print("Using workspace:", WORKSPACE)


Using workspace: C:\Users\Monil A\Desktop\Research Documents\ASSISTments Dataset


In [2]:
import pyarrow.parquet as pq
step2 = WORKSPACE / '02_Chronological_Learner_History'
spec = pd.read_csv(step2 / 'outputs' / 'ASSISTments_Step2_History_Feature_Specification.csv')
display(spec.groupby('history_configuration')['feature_name'].agg(['count', list]))
assert spec.groupby('history_configuration').size().to_dict() == {'H0':14, 'H1':15, 'H2':13}
print('Cumulative counts: H0=14, H1=29, H2=42')

,count,list
history_configuration,,
H0,14,"[skill_id, problem_type, tutor_mode, original,..."
H1,15,"[prior_interaction_count, prior_target_eligibl..."
H2,13,"[lag2_target_first_attempt_correct, lag3_targe..."


Cumulative counts: H0=14, H1=29, H2=42


In [3]:
history_file = step2 / 'data' / 'processed' / 'ASSISTments_2012_2013_history_features.parquet'
columns = ['user_id','start_time_parsed','problem_log_id','correct_raw','target_eligible',
           'target_first_attempt_correct','prior_interaction_count','previous_correct_raw',
           'prior_correct_rate','recent3_correct_rate','flag_timestamp_tie']
batch = next(pq.ParquetFile(history_file).iter_batches(batch_size=500, columns=columns)).to_pandas()
display(batch.head(12))
ordered = batch.sort_values(['user_id','start_time_parsed','problem_log_id']).reset_index(drop=True)
assert batch.reset_index(drop=True).equals(ordered)
print('Sample is in frozen chronological order: PASS')

,user_id,start_time_parsed,problem_log_id,correct_raw,target_eligible,target_first_attempt_correct,prior_interaction_count,previous_correct_raw,prior_correct_rate,recent3_correct_rate,flag_timestamp_tie
0,21421,2013-05-22 14:44:20,145035021,0.0,True,0,0,NaN,NaN,NaN,False
1,21421,2013-05-22 14:57:15,145035378,0.0,True,0,1,0.0,0.000000,0.000000,False
2,21421,2013-05-22 14:58:06,145035411,1.0,True,1,2,0.0,0.000000,0.000000,False
3,21421,2013-05-22 14:58:30,145035419,0.0,True,0,3,1.0,0.333333,0.333333,False
4,21421,2013-06-05 13:00:06,145281682,1.0,True,1,4,0.0,0.250000,0.333333,False
5,21421,2013-06-05 13:00:14,145281685,1.0,True,1,5,1.0,0.400000,0.666667,False
6,21421,2013-06-05 13:00:22,145281693,1.0,True,1,6,1.0,0.500000,0.666667,False
7,21421,2013-06-05 13:00:39,145281697,1.0,True,1,7,1.0,0.571429,1.000000,False
8,21421,2013-06-05 13:00:45,145281698,1.0,True,1,8,1.0,0.625000,1.000000,False
9,21421,2013-06-05 13:00:50,145281700,1.0,True,1,9,1.0,0.666667,1.000000,False


Sample is in frozen chronological order: PASS


In [4]:
tie_rows = batch[batch['flag_timestamp_tie'].fillna(False)]
print('Tie rows in this first batch:', len(tie_rows))
print('Pipeline rule: all peers sharing the current start timestamp are excluded from predictive history; problem_log_id is only a deterministic file-order tie-breaker.')
display(spec[['feature_name','history_configuration','mathematical_definition','timestamp_tie_behavior','leakage_risk']])

Tie rows in this first batch: 0
Pipeline rule: all peers sharing the current start timestamp are excluded from predictive history; problem_log_id is only a deterministic file-order tie-breaker.


,feature_name,history_configuration,mathematical_definition,timestamp_tie_behavior,leakage_risk
0,skill_id,H0,Current-row pre-response context field `skill_...,Unaffected by history; tie is flagged,Current/future inclusion; controlled by strict...
1,problem_type,H0,Current-row pre-response context field `proble...,Unaffected by history; tie is flagged,Current/future inclusion; controlled by strict...
2,tutor_mode,H0,Current-row pre-response context field `tutor_...,Unaffected by history; tie is flagged,Current/future inclusion; controlled by strict...
3,original,H0,Current-row pre-response context field `origin...,Unaffected by history; tie is flagged,Current/future inclusion; controlled by strict...
4,position,H0,Current-row pre-response context field `positi...,Unaffected by history; tie is flagged,Current/future inclusion; controlled by strict...
5,template_id,H0,Current-row pre-response context field `templa...,Unaffected by history; tie is flagged,Current/future inclusion; controlled by strict...
6,previous_correct_raw,H0,Shifted previous-event or accumulated learner ...,All same-timestamp peers excluded,Current/future inclusion; controlled by strict...
7,previous_attempt_count,H0,Shifted previous-event or accumulated learner ...,All same-timestamp peers excluded,Current/future inclusion; controlled by strict...
8,previous_ms_first_response,H0,Shifted previous-event or accumulated learner ...,All same-timestamp peers excluded,Current/future inclusion; controlled by strict...
9,previous_skill_id,H0,Shifted previous-event or accumulated learner ...,All same-timestamp peers excluded,Current/future inclusion; controlled by strict...
